# 04 · Continued pretraining

**The decision this notebook supports:** can I put new facts *into the weights*, rather than into the context window?

This notebook is deliberately brisk. Two things are worth slowing down for: the loss mask, and the honest answer about whether CPT is the right tool here at all.

**CPT trains on raw text with a next-token objective.** No instructions, no answers, no masking of anything. Every token contributes to the loss. That is the difference from SFT in the next notebook, and it is the whole difference.

**Precision:** CPT scores raw-text next-token predictions, including real document-ending EOS tokens. Padding is excluded from loss, and the first token of a window supplies context rather than its own target. SFT also changes the training examples and formatting; assistant-only masking is one important difference, not the only possible difference.

In [ ]:
# Load this project's shared paths, configuration, scoring, and file helpers.
# sys controls Python's import search path; adding '..' assumes the notebook folder
# is the working directory. Import the project root, not src itself, so src/secrets.py
# does not replace Python's own secrets module. warnings controls diagnostic messages;
# the existing filter hides warnings, which can also hide useful compatibility clues.
# json handles JSON data; it is imported here for later cells (unused in this setup).
# CFG is a nested dictionary: CFG["model"]["name"] reads its model-name setting.
import sys, json, warnings
sys.path.insert(0, "..")          # project root, so `from src import lab` works
warnings.filterwarnings("ignore") # hide library deprecation noise
from src import lab
CFG = lab.cfg()
print("lab ready |", CFG["model"]["name"])

In [ ]:
# Load the tokenizer (text <-> token IDs) and the pretrained next-token model.
# transformers supplies AutoTokenizer and AutoModelForCausalLM: the Auto classes
# choose the matching implementation from the saved checkpoint configuration.
# torch supplies tensors and GPU execution. float32 uses four bytes per weight;
# .to(DEVICE) moves the weights to Apple's MPS backend, explicitly selected here.
# This loads the original base; it does not resume the earlier CPT adapter.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
MODEL_PATH, DEVICE = CFG["model"]["base_path"], lab.device()
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
# If no padding token is configured, reuse EOS as its ID. Real EOS positions
# must still be trained; later masks distinguish padding by position, not ID.
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.float32).to(DEVICE)
# parameters() yields weight tensors; numel() counts their values. Dividing by
# 1e6 reports millions of parameters. Loading alone does not train the model.
print("loaded:", sum(p.numel() for p in model.parameters())/1e6, "M params on", DEVICE)

## Build token windows

Concatenate the corpus, tokenise once, then cut into fixed-length windows. Labels equal inputs, because the model predicts each token from the ones before it and the shift happens inside the loss function.

### Use the shipped CPT corpus

The code reads `data/generated/cpt_session/train.jsonl` and `validation.jsonl`. Each row supplies raw prose in `text`; metadata such as source fact IDs is for checking the data, not text to train on. These documents deliberately repeat the same facts in different wording. **Validation reserves document formats and recombinations; the same facts and many sentences occur in both splits.** No evaluation questions are used as training documents.

In [ ]:
# Read and validate the shipped training/validation documents.
# Only each row's text becomes training input; IDs and fact metadata audit provenance.
# read the two document splits shipped in data/generated/cpt_session.
# json parses each JSONL line independently. pathlib.Path is a filesystem helper
# (imported here but lab.GEN already supplies a Path). torch provides tensors,
# and torch.utils.data.Dataset defines the interface used by the trainer later.
import json
from pathlib import Path
import torch
from torch.utils.data import Dataset

CPT_DATA = lab.GEN / "cpt_session"

# A helper centralizes file/schema checks so malformed data fails before training.
# The list comprehension skips blank lines and parses each remaining JSON object.
def read_documents(filename):
    path = CPT_DATA / filename
    if not path.is_file():
        raise FileNotFoundError(f"Missing CPT corpus: {path}")
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    if not rows:
        raise ValueError(f"No documents in {path}")
    for row in rows:
        if not isinstance(row.get("id"), str) or not row["id"]:
            raise ValueError(f"Each document needs a non-empty string id: {path}")
        if not isinstance(row.get("text"), str) or not row["text"].strip():
            raise ValueError(f"Document {row['id']} has no text")
    # A set comprehension removes duplicates. A smaller set than the row count
    # means at least two records share an ID, so the code raises an explicit error.
    if len({row["id"] for row in rows}) != len(rows):
        raise ValueError(f"Duplicate document IDs in {path}")
    return rows


In [ ]:

# Load both splits, then check that neither document IDs nor normalized full texts
# overlap. The & operator intersects sets; 'assert not' requires an empty intersection.
train_docs = read_documents("train.jsonl")
validation_docs = read_documents("validation.jsonl")
assert not ({row["id"] for row in train_docs} & {row["id"] for row in validation_docs})
# lambda creates a short function: collapse whitespace and normalize letter case.
# This catches duplicate full documents, not shared sentences or facts; those are
# deliberately present in both splits.
normalised_text = lambda row: " ".join(row["text"].split()).casefold()
assert not ({normalised_text(row) for row in train_docs}
            & {normalised_text(row) for row in validation_docs}), "Exact text overlaps splits"

print(f"{len(train_docs):,} train documents | {len(validation_docs):,} validation documents")
print("First document:", train_docs[0]["id"])
print(train_docs[0]["text"])

### Turn each document into raw-text tokens, then add its boundary

CPT uses **no chat template, questions, or assistant-only mask**. Tokenize each document without automatically inserted special tokens and append one EOS token. Pack training and validation separately, so no window crosses the split.

EOS marks where a document ends; it does not create an attention barrier. A packed model can attend to earlier documents in its window. We overlap consecutive windows by **one context token**: that keeps the prediction across each window boundary, without scoring the overlapping token twice.

In [ ]:
# Convert raw documents to token streams, retaining an EOS boundary after each.
# This uses no chat template: CPT learns next-token predictions in source prose.
# The train and validation streams are built independently so windows never mix splits.
if tok.eos_token_id is None or tok.pad_token_id is None:
    raise ValueError("CPT needs an EOS token and a padding token")


def tokenize_documents(rows):
    token_ids = []
    for row in rows:
        # Disable automatic special tokens; append exactly one EOS ourselves afterward.
        # extend adds all document IDs to the list; append adds one boundary ID.
        # EOS marks an ending but does not prevent attention to earlier packed documents.
        document_ids = tok(row["text"], add_special_tokens=False)["input_ids"]
        token_ids.extend(document_ids)
        token_ids.append(tok.eos_token_id)  # a REAL boundary, included in loss
    return token_ids


train_ids = tokenize_documents(train_docs)
validation_ids = tokenize_documents(validation_docs)
L = CFG["training"]["cpt"]["max_seq_len"]


# A window needs context and at least one target. The list comprehension slices
# [start:start+max_length], excluding the end index; the final slice may be shorter.
# range's step is L-1, so adjacent windows overlap by one context token.
def make_windows(token_ids, max_length):
    if max_length < 2 or len(token_ids) < 2:
        raise ValueError("A causal window needs at least two real tokens")
    # Stride L-1 means adjacent windows share exactly one context token.
    # Stop before a one-token remainder: that token was already scored in the prior window.
    return [token_ids[start:start + max_length]
            for start in range(0, len(token_ids) - 1, max_length - 1)]


In [ ]:


# Build windows for both splits. Each window supplies len(w)-1 causal targets:
# the model predicts positions 1 onward from their preceding inputs. These asserts
# check that the stream's N-1 transitions are covered once, including the tail.
windows = make_windows(train_ids, L)
validation_windows = make_windows(validation_ids, L)
assert sum(len(w) - 1 for w in windows) == len(train_ids) - 1
assert sum(len(w) - 1 for w in validation_windows) == len(validation_ids) - 1
print(f"Train: {len(train_ids):,} tokens → {len(windows):,} windows")
print(f"Validation: {len(validation_ids):,} tokens → {len(validation_windows):,} windows")
# [-1] selects the last window, which can contain fewer than L real tokens.
# Padding that window is separate from deciding which tokens are training targets.
print(f"Window length: {L}; stride: {L-1}; final train window: {len(windows[-1])} real tokens")

### Pad the last window, and inspect what contributes to loss

The trainer receives equal-length tensors. Real tokens have attention mask `1`; padding has `0`. Real-token labels copy the inputs, while padding labels are `-100` and are ignored by cross-entropy. **Mask by position, not by token ID**: a real EOS must remain a target even if EOS is also used for padding.

Hugging Face's causal model shifts logits and labels internally. Do not shift these labels again. Each window needs at least two real tokens to contain a prediction.

In [ ]:
# Wrap token windows in PyTorch's Dataset interface and pad them to length L.
# __len__ tells the trainer how many examples exist; __getitem__ builds one example.
# self stores each dataset's windows, length, and pad ID. No gradients or training
# occur while constructing these integer input/label tensors.
class CPTData(Dataset):
    def __init__(self, token_windows, max_length, pad_id):
        self.windows = token_windows
        self.max_length = max_length
        self.pad_id = pad_id

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):
        # Token IDs are integer indices, so use torch.long, not floating-point values.
        # (self.max_length,) is a one-element shape tuple describing a 1-D tensor.
        real = torch.tensor(self.windows[index], dtype=torch.long)
        n_real = len(real)
        assert 2 <= n_real <= self.max_length
        input_ids = torch.full((self.max_length,), self.pad_id, dtype=torch.long)
        # Fill the leading real-token positions; the remainder keeps the padding ID.
        # The attention mask is 1 on real inputs and 0 on padding, telling the model
        # which positions contain usable context.
        input_ids[:n_real] = real
        attention_mask = torch.zeros(self.max_length, dtype=torch.long)
        attention_mask[:n_real] = 1
        # clone creates separate storage before changing labels. Setting padding to -100
        # tells cross-entropy to ignore those targets; attention_mask alone would not do that.
        # The causal LM shifts predictions/targets internally, so do not shift labels here.
        labels = input_ids.clone()
        labels[n_real:] = -100  # ignore PADDING, preserving every real EOS
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


In [ ]:


# Construct both datasets and inspect their final padded example before training.
# A boolean mask selects positions; ~real_mask inverts it to select padding.
ds = CPTData(windows, L, tok.pad_token_id)
validation_ds = CPTData(validation_windows, L, tok.pad_token_id)
example = ds[len(ds) - 1]
real_mask = example["attention_mask"].bool()
assert torch.equal(example["labels"][real_mask], example["input_ids"][real_mask])
# all() requires every padding label to be ignored. Equal-ID real EOS tokens
# remain supervised because masking uses positions rather than token identity.
assert (example["labels"][~real_mask] == -100).all()
print("Last train window real tokens:", int(real_mask.sum()))
print("Padding positions excluded from loss:", int((~real_mask).sum()))
# [1:] excludes the first input position: it is context, not a predicted target
# inside this window. .sum() counts remaining nonignored labels; int prints a scalar.
print("Scored next-token targets:", int((example["labels"][1:] != -100).sum()))
print("Decoded real-token tail:", repr(tok.decode(example["input_ids"][real_mask][-60:].tolist())))

## The loss, by hand, before the trainer

Cross-entropy on one token so the number in the training log means something to you.

In [ ]:
# Compute one next-token cross-entropy by hand so the training loss has meaning.
# torch.nn.functional, shortened to F, provides stateless neural-network operations.
# Logits are unnormalized scores shaped [one example, three vocabulary entries];
# target [0] selects the first entry as correct. This is a toy vocabulary, not Qwen.
import torch.nn.functional as F
logits = torch.tensor([[2.0, 1.0, 0.1]])   # 3-token vocabulary
target = torch.tensor([0])
# softmax normalizes across the last dimension (-1), giving probabilities that
# sum to one. Cross-entropy expects logits and applies the log-softmax internally;
# feeding probabilities to cross_entropy would compute a different loss.
probs = F.softmax(logits, dim=-1)
print("probabilities:", probs.tolist()[0])
print("manual  -log p(correct):", round(-torch.log(probs[0,0]).item(), 4))
print("F.cross_entropy        :", round(F.cross_entropy(logits, target).item(), 4))
# print("\nA loss of ~1.1 means the model gives the right token about", round(float(probs[0,0])*100), "% probability.")
# .item() extracts a Python scalar from a one-element tensor. For this single
# target, loss=-log(p_correct), hence p_correct=exp(-loss). Average training loss
# instead averages many targets; exp(-average loss) is their geometric mean probability.
actual_loss = F.cross_entropy(logits, target).item()
print(f"\nHere the loss is {actual_loss:.4f}, so p(correct) = exp(-loss) = {probs[0, 0].item():.3f}.")
print("A loss of 1.1 would mean exp(-1.1) ≈ 0.333, not the probability above.")

## Attach LoRA and train

Only the adapter matrices update. The base weights stay frozen, which is what makes this fit in memory.

The epochs, learning rate, training batch size, accumulation and LoRA settings still come from the existing config. The installed Transformers version selects the available device automatically. We evaluate validation loss once per epoch. This checks fit to reserved document formats and recombinations; the same facts and many sentences occur in both splits. It does not measure factual correctness or unseen knowledge.

In [ ]:
# PEFT means parameter-efficient fine-tuning: it adds trainable low-rank adapters
# while keeping the base weights frozen. LoraConfig describes the adapter;
# get_peft_model attaches it to the loaded model. r sets rank, alpha sets the
# standard alpha/r scaling, dropout regularizes adapter inputs, and target_modules
# selects which projection layers receive adapters. CAUSAL_LM selects next-token use.
# print_trainable_parameters reports actual target shapes, unlike the toy estimate.
from peft import LoraConfig, get_peft_model
lc = CFG["training"]["lora"]
peft_cfg = LoraConfig(r=lc["r"], lora_alpha=lc["alpha"], lora_dropout=lc["dropout"],
                      target_modules=lc["target_modules"], task_type="CAUSAL_LM")
m = get_peft_model(model, peft_cfg)
m.print_trainable_parameters()

In [ ]:
# Configure and run adapter training with a separate validation dataset.
# Transformers TrainingArguments stores training choices; Trainer implements the
# forward pass, causal loss, backpropagation, optimizer updates, and validation loop.
# same training defaults, with a separate validation split.
from transformers import Trainer, TrainingArguments
c = CFG["training"]["cpt"]
# A batch contains per_device_train_batch_size windows. Gradient accumulation
# combines several microbatches before one optimizer update, approximating a larger
# batch with less memory. epochs repeats the training set; learning_rate controls
# update size. eval_strategy evaluates once per epoch without training on validation.
# save_strategy='no' avoids periodic checkpoints; the next cell explicitly saves.
# report_to=[] disables external tracker reporting; the seed controls supported randomness.
args = TrainingArguments(
    output_dir=str(lab.RUNS / "cpt_session"), num_train_epochs=c["epochs"],
    per_device_train_batch_size=c["batch_size"], gradient_accumulation_steps=c["grad_accum"],
    per_device_eval_batch_size=c["batch_size"],
    learning_rate=c["lr"], logging_steps=5, eval_strategy="epoch", save_strategy="no",
    report_to=[], seed=CFG["seeds"]["train"])


In [ ]:
# Construct the trainer around the LoRA-wrapped model and the two datasets.
# The active Transformers version selects its available device automatically;
# printing args.device confirms what Trainer will actually use.
trainer = Trainer(model=m, args=args, train_dataset=ds, eval_dataset=validation_ds)
print("Trainer device:", args.device)
# This is the call that performs training and changes adapter weights.
# Validation loss measures next-token fit to the reserved documents, not factual
# answer accuracy or knowledge absent from the repeated training facts.
res = trainer.train()
print(res.metrics)

In [ ]:
# Persist the trained adapter and the evidence needed to identify this run.
# Artifacts use the cpt_session name: adapters/cpt_session and runs/06_cpt_session.json.
# hashlib is Python's cryptographic-digest library. SHA-256 fingerprints each input
# file's bytes so later readers can tell whether the corpus changed.
# separate session artifacts leave earlier runs untouched.
import hashlib
CPT_ADAPTER_PATH = lab.ADAPTERS / "cpt_session"
# PEFT saves the adapter configuration and trainable adapter weights, not a fresh
# copy of the frozen base. Reload it later alongside the compatible base checkpoint.
m.save_pretrained(str(CPT_ADAPTER_PATH))
# The dictionary comprehension maps each filename to its hexadecimal digest.
# This is a content identity check, not encryption or proof of factual correctness.
source_files = {name: hashlib.sha256((CPT_DATA / name).read_bytes()).hexdigest()
                for name in ["train.jsonl", "validation.jsonl", "manifest.json"]}
# Save metrics, filtered validation log entries, token/window counts, settings,
# paths, and file fingerprints as JSON. Rerunning this named save updates that
# session artifact; validation_history keeps only records containing eval_loss.
lab.save_run("06_cpt_session", {
    "metrics": res.metrics,
    "validation_history": [row for row in trainer.state.log_history if "eval_loss" in row],
    "train_documents": len(train_docs), "validation_documents": len(validation_docs),
    "train_tokens": len(train_ids), "validation_tokens": len(validation_ids),
    "train_windows": len(windows), "validation_windows": len(validation_windows),
    "max_seq_len": L, "window_stride": L - 1,
    "training_config": c, "lora_config": CFG["training"]["lora"],
    "corpus_path": str(CPT_DATA), "source_sha256": source_files,
    "adapter_path": str(CPT_ADAPTER_PATH),
    "validation_scope": "Reserved document formats and recombinations; the same facts and many sentences occur in both splits."
})
print("Session adapter saved to", CPT_ADAPTER_PATH)

## Did it work, and is that the right question?

Ask the tuned model a Meridian question with **no documents in context**. That is the only test that isolates what CPT actually bought you.

This single probe is a quick qualitative check. It has a different system prompt and token limit from notebook 02; use the optional comparison below for matched dev-set conditions.

**Where this checkpoint lives:** the new adapter is `adapters/cpt_session`, and its training metadata is `runs/06_cpt_session.json`. Notebook 05 intentionally starts from the original base model. Downstream notebooks do not automatically load this adapter; in particular, notebook 08 currently has no CPT selection path. Do not assume that setting its `BEST` string to `cpt` or `cpt_session` would load these weights.

In [ ]:
# Probe the trained adapter without any company documents in the prompt.
# eval() disables dropout; torch.no_grad() separately avoids gradient bookkeeping.
# The helper's mdl argument makes the model being queried explicit. This is a
# single qualitative probe with a 100-token cap, not the matched baseline experiment.
m.eval()  # disable training-time dropout before this qualitative generation

def ask(mdl, q, system="You are the Meridian Motors assistant."):
    msgs=[{"role":"system","content":system},{"role":"user","content":q}]
    # Render the saved chat template, including the assistant-start marker, then
    # convert it to tensors on the same device as the model. **i unpacks those tensors
    # as named input_ids/attention_mask arguments for greedy generation.
    t = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    i = tok(t, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        o = mdl.generate(**i, max_new_tokens=100, do_sample=False, pad_token_id=tok.eos_token_id)
    # Take batch row zero and remove the prompt prefix using its token length.
    # Decode only the new answer IDs; control tokens and outer whitespace are removed.
    return tok.decode(o[0][i["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(ask(m, "What is the certified range of the Meridian Arc 110?"))

## Optional: compare CPT with the same no-context baselines

Set `RUN_CPT_DEV = True` only when you want to run these generations. This evaluates **development questions only**, in two conditions matching notebook 02: its naive system prompt and its engineered system prompt, each with no documents, greedy decoding, a 3,072-token input cap and **160 new tokens**. With 36 dev questions, both conditions require 72 generations.

The saved stock runs supply the before-training comparison; this cell does not regenerate them or overwrite them. It checks that their question IDs match the current dev set. The shared scorer is deliberately simple: numeric matching and refusal markers can misjudge answers, particularly colour lists. Inspect raw answers before treating a metric change as conclusive.

CPT may improve familiar factual recall, but fluent answers are not evidence of grounding. Raw policy prose in this corpus also does not substitute for SFT refusal examples or a runtime source-checking system. The final held-out evaluation remains reserved for notebook 08.

In [ ]:
# Optional local evaluation: the flag prevents all 72 generations unless enabled.
# Keep the entire if block together because its variables/functions depend on the
# same opt-in path. No training or API call occurs in this evaluation block.
RUN_CPT_DEV = False  # opt in: two full dev conditions, no API calls

if RUN_CPT_DEV:
    # tqdm.auto shows per-question progress; pandas builds the comparison table.
    # Both imports occur only when the optional evaluation is enabled.
    from tqdm.auto import tqdm
    import pandas as pd

    CPT_NAIVE = "You are a helpful assistant."
    CPT_ENGINEERED = (
        "You are the customer assistant for Meridian Motors, an electric two-wheeler company. "
        "Answer only from Meridian's official documents. If the documents do not contain the "
        "answer, say plainly that you do not have that information and offer to connect the "
        "customer to a dealer. Never guess a number. Never round a range figure upward."
    )
    # Match notebook02's naive/engineered prompts, greedy decoding, and output cap.
    # These conditions have no documents, testing recall/behavior in the weights.
    CPT_MAX_NEW_TOKENS = 160
    dev = lab.eval_set("dev")
    fidx = lab.fact_index()
    saved_runs = lab.load_runs()
    comparison = {}
    m.eval()

    # Render and tokenize the exact condition/question, then generate with the CPT
    # adapter in evaluation mode. truncation=True retains the earlier notebook's
    # 3,072-token truncating input cap; slicing removes the prompt from the output.
    def cpt_dev_answer(system, question):
        messages = [{"role": "system", "content": system},
                    {"role": "user", "content": question}]
        prompt_text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt_text, return_tensors="pt", truncation=True, max_length=3072).to(DEVICE)
        with torch.no_grad():
            output = m.generate(**inputs, max_new_tokens=CPT_MAX_NEW_TOKENS,
                                do_sample=False, pad_token_id=tok.eos_token_id)
        return tok.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # Each tuple holds a condition label, its system prompt, and the saved stock-run
    # key. Tuple unpacking in the loop names those three values without extra indexing.
    conditions = [
        ("naive", CPT_NAIVE, "01_stock_naive"),
        ("engineered", CPT_ENGINEERED, "02_stock_engineered"),
    ]
    for condition, system, baseline_key in conditions:
        scores, answers = [], []
        # Generate each dev answer once, score it with the shared heuristic, and retain
        # the full text for inspection. The f-string labels the progress bar by condition.
        for question in tqdm(dev, desc=f"CPT session / {condition}"):
            answer = cpt_dev_answer(system, question["question"])
            scores.append(lab.score_answer(answer, question, fidx))
            answers.append({"id": question["id"], "question": question["question"],
                            "answer": answer, "answerable": question["answerable"],
                            "expected": question["expected_value"]})
        summary = lab.summarise(scores)
        baseline = saved_runs.get(baseline_key)
        # The set comparison checks question-ID coverage before displaying a stock
        # comparison. It does not verify old prompt text, decoding, or model-file hashes,
        # so this is a limited compatibility check rather than full experiment provenance.
        matching_ids = bool(baseline) and ({a["id"] for a in baseline.get("answers", [])}
                                          == {q["id"] for q in dev})
        # Write separate CPT result files containing the prompt/settings and adapter path.
        # The source stock files are read only. save_run overwrites this CPT condition's
        # existing file if you deliberately rerun it.
        lab.save_run(f"06_cpt_session_eval_{condition}", {
            "summary": summary, "answers": answers,
            "system_prompt": system, "context": "none", "split": "dev",
            "max_new_tokens": CPT_MAX_NEW_TOKENS, "max_input_tokens": 3072,
            "do_sample": False, "base_model": MODEL_PATH,
            "adapter_path": str(lab.ADAPTERS / "cpt_session"),
            "baseline_run": baseline_key, "baseline_question_ids_match": matching_ids,
        })
        if matching_ids:
            comparison[f"stock / {condition}"] = baseline["summary"]
        else:
            print(f"No matching saved {baseline_key} baseline; CPT result saved without a baseline comparison.")
        comparison[f"CPT session / {condition}"] = summary

    columns = ["accuracy_answerable", "hallucination_rate_answerable",
               "over_refusal_rate", "appropriate_refusal_rate"]
    # .T transposes the DataFrame so run names become rows; [columns] selects the
    # metrics to compare. Refusal and value matching are approximate: inspect answers,
    # including colours and Arc's unsupported-fast-charging sentinel, before concluding.
    display(pd.DataFrame(comparison).T[columns])
    print("Read the saved answers as well as the scores; these are dev results, not a held-out claim.")
else:
    print("Optional dev comparison skipped. Set RUN_CPT_DEV = True to run it after CPT training.")

## The honest read

On a corpus this small, CPT usually produces **fluent-sounding recall with unreliable numbers**. It teaches the model the shape and vocabulary of the domain more than the specific figures.

That is not a failure of your setup, it is what CPT is for, and it is the reason the industry standard for factual question answering is retrieval rather than continued pretraining. Write down what you actually observed before moving on.

**Next:** notebook 05 does supervised fine-tuning, which teaches behaviour rather than facts.

**Interpret your results:** repeated facts may improve recall, but validation loss measures text prediction. It does not establish correct numbers, reliable refusals, or generalization to unseen facts. The optional matched dev comparison above tests those behaviours more directly, within the limits of its simple scorer.